# rizzvision-v2: EfficientNetB3 T-Shirt Classifier (Google Colab)

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 **Secrets** icon (left sidebar) → Add new secret
   - Name: `KAGGLE_TOKEN` · Value: your `KGAT_...` token · toggle **Notebook access** ON
3. Run all cells

**Outputs:** `tshirt_classifier.keras` + `threshold.json` saved to `/content/`

In [ ]:
# ── 1. Kaggle API setup + dataset download ────────────────────────────────────
# Reads your token from Colab Secrets (Key: KAGGLE_TOKEN).
# To add it: click the 🔑 Secrets icon in the left sidebar → Add new secret
#   Name:  KAGGLE_TOKEN
#   Value: your token (the KGAT_... string)
#   Toggle 'Notebook access' ON
import os
from google.colab import userdata

token = userdata.get('KAGGLE_TOKEN')
os.environ['KAGGLE_TOKEN'] = token

!pip install -q kaggle
!kaggle datasets download -d rizzvision/rizzvision-tshirt-dataset -p /content/data --unzip

print('\nDownloaded. Top-level contents:')
for item in sorted(os.listdir('/content/data')):
    print(' ', item)

In [ ]:
# ── 2. Imports & config ────────────────────────────────────────────────────────
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, confusion_matrix

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# ── Paths ──────────────────────────────────────────────────────────────────────
OUT_DIR = '/content'

# Dataset was downloaded + unzipped to /content/data by the cell above.
# Adjust the path below if the zip contained a subfolder (the printout will show you).
# Expected structure:
#   DATASET_DIR/
#     train/  <class_a>/  <class_b>/
#     val/    <class_a>/  <class_b>/
#     test/   <class_a>/  <class_b>/
DATASET_DIR = '/content/data'  # change if needed, e.g. '/content/data/rizzvision-tshirt-dataset'

print('Dataset dir:', DATASET_DIR)
print('Contents:', os.listdir(DATASET_DIR))

INPUT_SIZE = 300
BATCH_SIZE = 32
AUTOTUNE  = tf.data.AUTOTUNE

In [ ]:
# ── 3. Dataset pipeline ────────────────────────────────────────────────────────
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 50.0)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    crop_size = tf.random.uniform(
        [], minval=int(INPUT_SIZE * 0.85), maxval=INPUT_SIZE, dtype=tf.int32
    )
    image = tf.image.random_crop(image, size=[crop_size, crop_size, 3])
    image = tf.image.resize(image, [INPUT_SIZE, INPUT_SIZE])
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def load_dataset(split, augment_data=False):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATASET_DIR, split),
        image_size=(INPUT_SIZE, INPUT_SIZE),
        batch_size=None,
        label_mode='binary',
        shuffle=(split == 'train'),
        seed=42,
    )
    # EfficientNetB3 expects [0, 255] — cast only, do NOT normalize
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

train_ds = load_dataset('train', augment_data=True)
val_ds   = load_dataset('val')
test_ds  = load_dataset('test')
print('Datasets loaded')

In [ ]:
# ── 4. Build model ─────────────────────────────────────────────────────────────
def build_model(trainable_base=False):
    base = keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    )
    base.trainable = trainable_base
    inputs  = keras.Input(shape=(INPUT_SIZE, INPUT_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs, outputs), base

model, base = build_model(trainable_base=False)
model.summary()

In [ ]:
# ── 5. Phase 1: train head only (5 epochs) ────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.BinaryFocalCrossentropy(gamma=2.0),
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/best_phase1.keras',
            save_best_only=True, monitor='val_auc', mode='max'
        ),
    ],
)
print('Phase 1 complete')

In [ ]:
# ── 6. Phase 2: fine-tune top 50 layers (up to 25 epochs) ─────────────────────
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=keras.losses.BinaryFocalCrossentropy(gamma=2.0),
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/tshirt_classifier.keras',
            save_best_only=True, monitor='val_auc', mode='max'
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=7, restore_best_weights=True
        ),
    ],
)
print('Phase 2 complete')

In [ ]:
# ── 7. Threshold optimisation on validation set ───────────────────────────────
best_model = keras.models.load_model(
    f'{OUT_DIR}/tshirt_classifier.keras', compile=False
)

val_probs, val_labels = [], []
for batch_x, batch_y in val_ds:
    val_probs.extend(best_model.predict(batch_x, verbose=0).flatten().tolist())
    val_labels.extend(batch_y.numpy().flatten().tolist())

val_probs  = np.array(val_probs)
val_labels = np.array(val_labels)
print(f'Val AUC: {roc_auc_score(val_labels, val_probs):.4f}')

best_thresh, best_f1 = 0.5, 0.0
neg_count = (val_labels == 0).sum()
pos_count = (val_labels == 1).sum()

for t in np.arange(0.10, 0.95, 0.01):
    preds    = (val_probs >= t).astype(int)
    tp_count = ((preds == 1) & (val_labels == 1)).sum()
    fp_count = ((preds == 1) & (val_labels == 0)).sum()
    fn_count = ((preds == 0) & (val_labels == 1)).sum()
    fpr_t    = fp_count / neg_count
    if fpr_t > 0.02:
        continue
    precision = tp_count / (tp_count + fp_count + 1e-9)
    recall    = tp_count / (tp_count + fn_count + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = float(t)

print(f'Optimal threshold (FPR <= 2%): {best_thresh:.4f}  (F1: {best_f1:.4f})')
with open(f'{OUT_DIR}/threshold.json', 'w') as f:
    json.dump({'threshold': round(best_thresh, 4)}, f, indent=2)

In [ ]:
# ── 8. Test set evaluation ────────────────────────────────────────────────────
test_probs, test_labels = [], []
for batch_x, batch_y in test_ds:
    test_probs.extend(best_model.predict(batch_x, verbose=0).flatten().tolist())
    test_labels.extend(batch_y.numpy().flatten().tolist())

test_probs  = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds  = (test_probs >= best_thresh).astype(int)

cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / len(test_labels)
fpr      = fp / (fp + tn)
fnr      = fn / (fn + tp)

print('=== Test Set Results ===')
print(f'Accuracy: {accuracy:.4f}  (target >= 0.93)')
print(f'AUC-ROC:  {roc_auc_score(test_labels, test_probs):.4f}  (target >= 0.97)')
print(f'FPR:      {fpr:.4f}  (target <= 0.02)')
print(f'FNR:      {fnr:.4f}  (target <= 0.08)')
print(f'Threshold used: {best_thresh}')
print('\nConfusion matrix:')
print(cm)

if accuracy >= 0.93 and fpr <= 0.02:
    print('\n✓ All targets met — model ready for deployment')
else:
    print('\n✗ Targets not met — retrain with more epochs')

In [ ]:
# ── 9. Save outputs ───────────────────────────────────────────────────────────
# Files are already in /content/ — download them via the Files panel
# (left sidebar → folder icon → right-click → Download)

for fname in ['tshirt_classifier.keras', 'threshold.json']:
    path = f'{OUT_DIR}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'✓ {fname}  ({size:.1f} MB)')
    else:
        print(f'✗ {fname} not found — check training completed successfully')

print('\nDownload via: Files panel (left sidebar) → right-click each file → Download')
print('Or run the cell below to also back them up to Google Drive.')

In [ ]:
# ── 10. (Optional) Back up outputs to Google Drive ────────────────────────────
# Run this if you want to keep the files after the Colab session ends.
import shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/rizzvision/output'
os.makedirs(DRIVE_OUT, exist_ok=True)

for fname in ['tshirt_classifier.keras', 'threshold.json']:
    src = f'{OUT_DIR}/{fname}'
    dst = f'{DRIVE_OUT}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Copied {fname} → {dst}')
    else:
        print(f'WARNING: {fname} not found')

print('Saved to Google Drive:', DRIVE_OUT)